# Code mẫu — Data split và cross-validation

Cách chia dữ liệu phải giống loại mẫu mà model sẽ gặp khi được dùng thật: một bản ghi mới, một người mới, hay một thời điểm mới. Các ví dụ dưới đây kiểm lần lượt lớp hiếm, nhóm, thời gian, và cách ghép dự đoán OOF về đúng hàng, trước khi đọc bất kỳ điểm số nào.

## Ví dụ 1: Validation có còn mẫu của lớp hiếm không?

Có mười hai bài làm, trong đó ba bài mang nhãn “cần gợi ý” (`1`) và chín bài mang nhãn `0`. Ta giữ lại ba bài làm validation, và dùng stratify để phần này vẫn có mặt lớp hiếm.

In [ ]:
import numpy as np
from math import comb
from sklearn.model_selection import train_test_split

y = np.array([1, 1, 1] + [0] * 9)
ids = np.arange(len(y))
train_ids, valid_ids = train_test_split(
    ids, test_size=3, stratify=y, random_state=31,
)
print("Số hàng train/valid:", len(train_ids), len(valid_ids))
print("Số nhãn 1 train/valid:", int(y[train_ids].sum()), int(y[valid_ids].sum()))
print("Trùng hàng:", len(set(train_ids) & set(valid_ids)))
print(f"Xác suất thiếu lớp 1 nếu chọn đều ba hàng: {comb(9, 3)/comb(12, 3):.3f}")

Số hàng train/valid: 9 3
Số nhãn 1 train/valid: 2 1
Trùng hàng: 0
Xác suất thiếu lớp 1 nếu chọn đều ba hàng: 0.382


Nhờ stratify, validation có đúng một bài nhãn 1, nên không mất hẳn lớp hiếm. Nếu chọn ngẫu nhiên ba hàng mà không stratify, xác suất cả ba đều là nhãn 0 là `84/220 ≈ 0.382`, tức gần bốn lần trong mười. Dù vậy, một bài nhãn 1 vẫn quá ít để ước lượng chất lượng một cách ổn định. Với tập nhỏ như vậy, cross-validation nhiều fold đáng tin hơn một lần chia.

## Ví dụ 2: Không trùng hàng, nhưng vẫn có thể trùng học sinh

Có sáu học sinh, mỗi người nộp hai bài. Ta cần đánh giá model trên những học sinh **chưa từng xuất hiện** trong train. Trước hết, ta cố ý chia mỗi người một bài sang mỗi phía, rồi so với cách chia bằng GroupKFold.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold

groups = np.repeat(["A", "B", "C", "D", "E", "F"], 2)
X = np.zeros((12, 1))
y = np.repeat([0, 1, 0, 1, 0, 1], 2)
row_train = np.arange(0, 12, 2)
row_valid = np.arange(1, 12, 2)
print("Split từng hàng, trùng hàng:", len(set(row_train) & set(row_valid)))
print("Split từng hàng, trùng người:", sorted(map(str, set(groups[row_train]) & set(groups[row_valid]))))

cv = GroupKFold(n_splits=3)
for fold, (train_ids, valid_ids) in enumerate(cv.split(X, y, groups), start=1):
    shared = set(groups[train_ids]) & set(groups[valid_ids])
    assert not shared
    valid_groups = sorted(map(str, set(groups[valid_ids])))
    print(f"Fold {fold}: valid={valid_groups}, nhãn 1={int(y[valid_ids].sum())}, trùng người={len(shared)}")

Split từng hàng, trùng hàng: 0
Split từng hàng, trùng người: ['A', 'B', 'C', 'D', 'E', 'F']
Fold 1: valid=['C', 'F'], nhãn 1=2, trùng người=0
Fold 2: valid=['B', 'E'], nhãn 1=2, trùng người=0
Fold 3: valid=['A', 'D'], nhãn 1=2, trùng người=0


Cách chia đầu tiên có **0 hàng trùng**, nghe có vẻ sạch, nhưng cả sáu học sinh đều có mặt ở hai phía. Model chỉ cần nhận ra cách viết của từng người là đã đoán tốt, nên điểm validation sẽ lạc quan. Với GroupKFold, hai bài của một người luôn nằm cùng một phía, nên validation thật sự là người mới. Nếu câu hỏi là dự đoán cho một trường mới, thì `groups` phải là mã trường, vì nhóm theo học sinh chưa trả lời được câu hỏi ấy.

## Ví dụ 3: Nhãn của train phải hai tuần sau mới biết

Mỗi hàng ở tuần t có target là kết quả của hai tuần sau, nên nhãn ấy chỉ được biết ở tuần `t + 2`. Dữ liệu đã được sắp theo tuần. Ta so `gap=0` với `gap=2`, để kiểm xem train có dùng tới những nhãn mà lúc bắt đầu validation vẫn chưa ai biết hay không.

In [ ]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

weeks = np.arange(12)
label_known_at = weeks + 2
for gap in [0, 2]:
    cv = TimeSeriesSplit(n_splits=3, test_size=2, gap=gap)
    for fold, (train_ids, valid_ids) in enumerate(cv.split(weeks), start=1):
        prediction_week = int(weeks[valid_ids].min())
        latest_label = int(label_known_at[train_ids].max())
        available = latest_label < prediction_week
        print(f"gap={gap}, fold={fold}: train cuối={int(weeks[train_ids].max())}, valid đầu={prediction_week}, nhãn biết lúc={latest_label}, hợp lệ={available}")
        if gap == 2:
            assert available

gap=0, fold=1: train cuối=5, valid đầu=6, nhãn biết lúc=7, hợp lệ=False
gap=0, fold=2: train cuối=7, valid đầu=8, nhãn biết lúc=9, hợp lệ=False
gap=0, fold=3: train cuối=9, valid đầu=10, nhãn biết lúc=11, hợp lệ=False
gap=2, fold=1: train cuối=3, valid đầu=6, nhãn biết lúc=5, hợp lệ=True
gap=2, fold=2: train cuối=5, valid đầu=8, nhãn biết lúc=7, hợp lệ=True
gap=2, fold=3: train cuối=7, valid đầu=10, nhãn biết lúc=9, hợp lệ=True


Ở `gap=0`, train kết thúc ở tuần 5 và validation bắt đầu ở tuần 6, trông có vẻ đúng thứ tự. Nhưng nhãn của tuần 5 phải tới tuần 7 mới biết, tức là train đang dùng một thông tin thuộc về tương lai. Chỉ nhìn tuần cuối của train là chưa đủ, phải nhìn cả lúc nhãn được biết. Với `gap=2`, nhãn mới nhất của train luôn được biết **trước** tuần dự đoán. Nếu lịch ghi nhận nhãn ở bài thật khác đi, phải tính lại điều kiện này và chọn gap tương ứng.

## Ví dụ 4: Ghi dự đoán OOF về đúng hàng gốc

Chín mẫu giả lập tuân theo `y = 2x + 1`. Ta chia ba fold có xáo trộn, rồi fit LinearRegression. Vì quan hệ không có nhiễu, dự đoán OOF đúng phải khớp y. Đây là phép kiểm cách ghép chỉ số, không phải báo cáo chất lượng model.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = np.arange(9, dtype=float).reshape(-1, 1)
y = 2 * X[:, 0] + 1
oof = np.full(len(y), np.nan)
count = np.zeros(len(y), dtype=int)
chunks = []
cv = KFold(n_splits=3, shuffle=True, random_state=32)
for fold, (train_ids, valid_ids) in enumerate(cv.split(X), start=1):
    model = LinearRegression().fit(X[train_ids], y[train_ids])
    prediction = model.predict(X[valid_ids])
    oof[valid_ids] = prediction       # Ghi vào đúng vị trí gốc, để mỗi dự đoán đứng cạnh nhãn của chính nó.
    count[valid_ids] += 1
    chunks.append(prediction)
    print(f"Fold {fold}: valid IDs={valid_ids.tolist()}")
assert np.all(count == 1)
assert np.isfinite(oof).all()
assert np.allclose(oof, y)
print(f"OOF đúng: MAE={mean_absolute_error(y, oof):.3f}")
print(f"Nối theo fold rồi ghép với y gốc, cách sai: MAE={mean_absolute_error(y, np.concatenate(chunks)):.3f}")

Fold 1: valid IDs=[1, 2, 6]
Fold 2: valid IDs=[0, 4, 8]
Fold 3: valid IDs=[3, 5, 7]
OOF đúng: MAE=0.000
Nối theo fold rồi ghép với y gốc, cách sai: MAE=4.000


Hai mảng dự đoán đều dài 9, nhưng chỉ cách gán `oof[valid_ids]` giữ đúng quan hệ giữa dự đoán và nhãn. Cách nối các phần theo thứ tự fold đặt dự đoán của hàng 1, 2, 6 vào vị trí 0, 1, 2, nên MAE nhảy từ 0 lên 4, dù model không hề tệ đi. Mảng `count` bằng 1 ở mọi hàng chứng tỏ không hàng nào bị chấm thiếu hay chấm hai lần. Với TimeSeriesSplit, những hàng đầu tiên không bao giờ vào validation, nên phải để trống và chấm qua mask, chứ không lấy dự đoán trên train để lấp vào.